# Day 5: Helpdesk data exploration

This notebook loads the seeded SQLite database into Pandas and uses the reusable analytics functions from `src/analytics/data_summary.py`. Run `python db/seed.py` first if the database does not exist.

In [1]:
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATABASE_PATH = PROJECT_ROOT / "db" / "helpdesk.sqlite3"
connection = sqlite3.connect(DATABASE_PATH)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
from analytics.data_summary import (
    engineer_workload,
    financial_summary,
    high_ticket_volume_accounts,
    ticket_distribution,
)

In [ ]:
tables = ["customers", "jobs", "invoices", "tickets"]
dataframes = {
    table: pd.read_sql_query(f"SELECT * FROM {table}", connection) for table in tables
}
customers = dataframes["customers"]
jobs = dataframes["jobs"]
invoices = dataframes["invoices"]
tickets = dataframes["tickets"]
{table: frame.shape for table, frame in dataframes.items()}

## Ticket distribution

In [ ]:
ticket_summary = ticket_distribution(tickets)
ticket_summary

In [ ]:
tickets.groupby(["category", "priority"]).size().unstack(fill_value=0).plot.bar(
    title="Tickets by category and priority", figsize=(9, 4)
);

## Financial analysis

In [ ]:
pending_invoices = invoices[invoices["status"].isin(["unpaid", "overdue"])]
financial_summary(invoices)

In [ ]:
average_invoice_amount = float(np.mean(invoices["amount"]))
total_overdue_revenue = float(pending_invoices["amount"].sum())
{
    "average_invoice_amount": average_invoice_amount,
    "total_overdue_revenue": total_overdue_revenue,
}

## Engineer workload

In [ ]:
engineer_workload(jobs)

## Multi-table account insights

In [ ]:
high_ticket_volume_accounts(tickets, customers, invoices)